# Robot cell: FANUC CR-35iA and UR20

Two FANUC CR-35iA and two UR20 work at two tables. The robot models come from the official
description packages (`fanuc_cr35ia_support` of ROS-Industrial, `ur_description` of Universal Robots).

- Layout: `ros/biba_cell_description/config/cell_layout.yaml`
- Motion: `ros/biba_cell_description/config/motions.yaml` (joint-space waypoints, repeated in a loop)

## Start the cell

Run the cell below, then click **Start cell**. RVizWeb opens on the right with a view of the whole cell.
Without *animate*, every robot holds its first waypoint. *speed* scales the motion.

In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from lab_launch import LaunchPanel

pkg = Path.home() / "work/ros/biba_cell_description"
animate = widgets.Checkbox(value=True, description="animate")
speed = widgets.FloatSlider(value=1.0, min=0.25, max=2.0, step=0.25, description="speed")

cell = LaunchPanel(
    "cell",
    lambda: ["biba_cell_description", "cell.launch",
             f"animate:={str(animate.value).lower()}", f"speed:={speed.value}"],
    rvizweb_config=pkg / "rviz/cell.json",
)
display(widgets.HBox([animate, speed]))
cell.show()

## Change the layout

Edit `cell_layout.yaml` and click **Start cell** again; no rebuild is needed.
The waypoints fit the delivered layout: a robot moved relative to its table can miss the table.

In [ ]:
import yaml

layout = yaml.safe_load((pkg / "config/cell_layout.yaml").read_text())
for r in layout["robots"]:
    print(f"{r['id']:8s} {r['type']:7s} x={r['x']:5.2f} y={r['y']:5.2f} yaw={r['yaw']:5.2f} "
          f"pedestal={r.get('pedestal', 0):.2f}")
for t in layout.get("tables", []):
    print(f"{t['id']:8s} table   x={t['x']:5.2f} y={t['y']:5.2f} size={t['size']}")

## What is running

`robot_state_publisher` computes the transforms, `waypoint_player` publishes the joint positions.

In [ ]:
!rosnode list
!timeout 4 rostopic hz /joint_states